In [35]:
from google.colab import userdata

APP_ID = userdata.get("ADZUNA_APP_ID")
APP_KEY = userdata.get("ADZUNA_APP_KEY")

print("APP_ID loaded: ", APP_ID is not None)
print("APP_KEY loaded: ", APP_KEY is not None)

APP_ID loaded:  True
APP_KEY loaded:  True


In [36]:
import requests
import pandas as pd
from datetime import datetime
import time

In [37]:
BASE_URL = "https://api.adzuna.com/v1/api/jobs/gb/search"

In [38]:
url = f"{BASE_URL}/1"

params = {
    "app_id": APP_ID,
    "app_key": APP_KEY,
    "what": "data analyst",
    "where": "UK",
    "results_per_page": 20,
    "content-type": "application/json"
}

response = requests.get(url, params=params)
response.raise_for_status()

data = response.json()



In [39]:
data.keys()

dict_keys(['mean', 'results', 'count', '__CLASS__'])

In [40]:
len(data["results"])

20

In [41]:
data["results"][0]

{'latitude': 53.798801,
 'company': {'__CLASS__': 'Adzuna::API::Response::Company',
  'display_name': 'BAE Systems'},
 'salary_max': 53889.32,
 'title': 'Lead Data Analyst/Project Controller',
 'longitude': -2.305057,
 'salary_min': 53889.32,
 '__CLASS__': 'Adzuna::API::Response::Job',
 'description': 'Job Title: Lead Data Analyst/Project Controller Location: Samlesbury. Hybrid – 3 days per week on site. We offer a range of hybrid and flexible working arrangements – please speak to your recruiter about the options for this particular role Salary: £59,491 DOE Who we are: Join BAE Systems and you’ll be part of something bigger. As a valued member of our global colleague network, you’ll bring your unique skills and perspectives to help pioneer progress and protect what matters most. You’ll be tr…',
 'redirect_url': 'https://www.adzuna.co.uk/jobs/land/ad/5840324082?se=hGzMNeyb8RGlrvMe-VN8Yg&utm_medium=api&utm_source=e24fca89&v=BD9028B403D572FE1BF1BE56C807F9A04FF2F40C',
 'category': {'label

In [42]:
jobs = []

for job in data["results"]:
  jobs.append({
      "job_id": job.get("id"),
      "title": job.get("title"),
      "company": job.get("company", {}).get("display_name"),
      "location": job.get("location", {}).get("display_name"),
      "created": job.get("created"),
      "description": job.get("description"),
      "salary_min": job.get("salary_min"),
      "salary_max": job.get("salary_max"),
      "salary_is_predicted": job.get("salary_is_predicted"),
      "latitude": job.get("latitude"),
      "longitude": job.get("longitude"),
      "redirect_url": job.get("redirect_url"),
      "contract_type": job.get("contract_type"),
      "contract_time": job.get("contract_time"),
      "category": job.get("category", {}).get("label")
  })
df = pd.DataFrame(jobs)

DATAFRAME

In [43]:
df.head()

,job_id,title,company,location,created,description,salary_min,salary_max,salary_is_predicted,latitude,longitude,redirect_url,contract_type,contract_time,category
0,5840324082,Lead Data Analyst/Project Controller,BAE Systems,"Padiham, Burnley",2026-08-13T15:06:21Z,Job Title: Lead Data Analyst/Project Controlle...,53889.32,53889.32,1,53.798801,-2.305057,https://www.adzuna.co.uk/jobs/land/ad/58403240...,None,None,IT Jobs
1,5840324089,Lead Data Analyst/Project Controller,BAE Systems,"Penwortham, Preston",2026-08-13T15:06:21Z,Job Title: Lead Data Analyst/Project Controlle...,53416.18,53416.18,1,53.758495,-2.701628,https://www.adzuna.co.uk/jobs/land/ad/58403240...,None,None,IT Jobs
2,5840324100,Lead Data Analyst/Project Controller,BAE Systems,"Samlesbury, Preston",2026-08-13T15:06:21Z,Job Title: Lead Data Analyst/Project Controlle...,63011.66,63011.66,1,53.754200,-2.580850,https://www.adzuna.co.uk/jobs/land/ad/58403241...,None,None,IT Jobs
3,5835920078,Data Analyst,VANRATH,"Antrim, County Antrim",2026-08-10T20:53:51Z,Data Analyst/Data Officer (Public Sector) - £3...,40000.00,40000.00,0,54.713402,-6.216760,https://www.adzuna.co.uk/jobs/land/ad/58359200...,None,None,IT Jobs
4,5829131501,Data Analyst,MCS Group,"Banbridge, County Down",2026-08-05T15:10:55Z,Data Analyst Banbridge | £30-33k *PLEASE NOTE:...,30000.00,30000.00,0,54.351002,-6.267020,https://www.adzuna.co.uk/jobs/land/ad/58291315...,None,None,IT Jobs


**PAGINATION**

In [44]:
def collect_jobs(search_term, pages=10):
    jobs = []

    for page in range(1, pages + 1):

        url = f"{BASE_URL}/{page}"

        params = {
            "app_id": APP_ID,
            "app_key": APP_KEY,
            "what": search_term,
            "where": "UK",
            "results_per_page": 20,
            "content-type": "application/json"
        }

        response = requests.get(url, params=params)
        response.raise_for_status()

        data = response.json()

        for job in data.get("results", []):
            jobs.append({
                "job_id": job.get("id"),
                "title": job.get("title"),
                "company": job.get("company", {}).get("display_name"),
                "location": job.get("location", {}).get("display_name"),
                "created": job.get("created"),
                "description": job.get("description"),
                "salary_min": job.get("salary_min"),
                "salary_max": job.get("salary_max"),
                "salary_is_predicted": job.get("salary_is_predicted"),
                "latitude": job.get("latitude"),
                "longitude": job.get("longitude"),
                "redirect_url": job.get("redirect_url"),
                "contract_type": job.get("contract_type"),
                "contract_time": job.get("contract_time"),
                "category": job.get("category", {}).get("label"),
                "search_term": search_term,
                "collection_date": datetime.now()
            })

        time.sleep(0.5)

    return pd.DataFrame(jobs)

In [45]:
data_analyst = collect_jobs("data analyst")
bi_analyst = collect_jobs("business intelligence analyst")
reporting_analyst = collect_jobs("reporting analyst")

In [46]:
df_raw = pd.concat ([
    data_analyst,
    bi_analyst,
    reporting_analyst
],
  ignore_index=True
                    )

In [47]:
df_raw.shape

(532, 17)

In [48]:
print("Total rows:", len(df_raw))
print("Unique job IDs:", df_raw["job_id"].nunique())
print(
    "Duplicate rows:",
    df_raw.duplicated(subset="job_id").sum()
)

Total rows: 532
Unique job IDs: 491
Duplicate rows: 41


In [49]:
len(df_raw)

532

In [50]:
duplicates = (
    df_raw[df_raw.duplicated("job_id", keep=False)]
    .sort_values("job_id")
)

In [51]:
duplicates[
    [
        "job_id",
        "title",
        "company",
        "search_term"
    ]
].head(20)

,job_id,title,company,search_term
511,5210480071,Data Analyst,wipro,reporting analyst
182,5210480071,Data Analyst,wipro,data analyst
305,5773388931,Data Analyst,GlobalData PLC,business intelligence analyst
173,5773388931,Data Analyst,GlobalData PLC,data analyst
470,5804909149,Data Analyst,Lumen Partners,reporting analyst
166,5804909149,Data Analyst,Lumen Partners,data analyst
302,5808583587,Data Analyst,EXPRESS SOLICITORS,business intelligence analyst
177,5808583587,Data Analyst,EXPRESS SOLICITORS,data analyst
164,5814132404,Data Analyst,AWD online,data analyst
490,5814132404,Data Analyst,AWD online,reporting analyst


In [52]:
df_raw.duplicated(
    subset="job_id"
).sum()

np.int64(41)

**Multiple search queries produced overlapping vacancies. Job IDs were therefore used to identify and remove duplicate observations before analysis.**

In [55]:
df_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 532 entries, 0 to 531
Data columns (total 17 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   job_id               532 non-null    object        
 1   title                532 non-null    object        
 2   company              531 non-null    object        
 3   location             532 non-null    object        
 4   created              532 non-null    object        
 5   description          532 non-null    object        
 6   salary_min           532 non-null    float64       
 7   salary_max           532 non-null    float64       
 8   salary_is_predicted  532 non-null    object        
 9   latitude             372 non-null    float64       
 10  longitude            372 non-null    float64       
 11  redirect_url         532 non-null    object        
 12  contract_type        325 non-null    object        
 13  contract_time        364 non-null  

In [56]:
df_raw.isna().sum()

,0
job_id,0
title,0
company,1
location,0
created,0
description,0
salary_min,0
salary_max,0
salary_is_predicted,0
latitude,160


In [57]:
(
    df_raw.isna().mean()
    .mul(100)
    .round(2)
    .sort_values(ascending=False)
)

,0
contract_type,38.91
contract_time,31.58
latitude,30.08
longitude,30.08
company,0.19
job_id,0.00
title,0.00
location,0.00
created,0.00
salary_is_predicted,0.00


In [58]:
df_raw[
    ["salary_min", "salary_max"]
].describe()

,salary_min,salary_max
count,532.000000,532.000000
mean,46624.072726,53047.617838
std,19436.494633,19558.333237
min,0.000000,450.000000
25%,30000.000000,41849.740000
50%,43011.290000,50000.000000
75%,55036.190000,59389.115000
max,148101.000000,175500.000000


In [59]:
df_raw["search_term"].value_counts()

,count
search_term,
data analyst,200
reporting analyst,200
business intelligence analyst,132
